# PREPARATION

Prepares one image for inference on the FPGA.

Images can be any size or aspect ratio (JPG, PNG, WebP). The notebook scales the image to 640×640 with gray padding. Results are best when objects are not too small in the picture. Very small or very wide images can reduce the number of detections.

In [ ]:
import json
import os

import numpy as np
from IPython.display import display
from PIL import Image, ImageOps

IMAGE_NAME = "crossroad.jpeg"

SIZE = 640
PAD_VALUE = 114

IMAGE_PATH = os.path.join("images", IMAGE_NAME)
FRAME_FILE = "frame_load.txt"
META_FILE = "frame_meta.json"

## READ IMAGE

The image is rotated according to its EXIF orientation, as phone photos often store rotation there.
Grayscale, CMYK, and transparent PNG images are converted to RGB. Transparent areas keep the color stored
in their pixels, which is often black.

In [ ]:
def load_image(path):
    """Return the image as an RGB array (H x W x 3, uint8)."""
    img = ImageOps.exif_transpose(Image.open(path))
    return np.asarray(img.convert("RGB"))

In [ ]:
img = load_image(IMAGE_PATH)
orig_h, orig_w = img.shape[:2]
print(f"{IMAGE_NAME}: {orig_w} x {orig_h}")

In [ ]:
preview = Image.fromarray(img)
preview.thumbnail((320, 320))
display(preview)

## PROCESSING IMAGE BEFORE INFERENCE

**Letterbox.** The image is scaled by `r = min(640 / width, 640 / height)`, so the longer side becomes 640 and the
aspect ratio is kept. Scaling uses bilinear interpolation. The scaled image is placed in the center of a
640×640 gray canvas. Example: 1920×1080 becomes 640×360 with 140 rows of padding above and below.

This follows the YOLOv5 letterbox with a fixed square input. When comparing with a reference model on a PC,
the reference must also use a 640×640 letterbox, not the minimal padding that `detect.py` uses by default.

In [ ]:
def resize_linear(a, new_w, new_h):
    """Bilinear resize with half-pixel centers (same convention as cv2.INTER_LINEAR)."""
    h0, w0 = a.shape[:2]

    def axis(n_out, n_in):
        f = (np.arange(n_out) + 0.5) * (n_in / n_out) - 0.5
        i0 = np.floor(f).astype(int)
        t = np.where(i0 < 0, 0.0, f - i0)
        i0 = np.clip(i0, 0, n_in - 1)
        return i0, np.clip(i0 + 1, 0, n_in - 1), t

    y0, y1, ty = axis(new_h, h0)
    x0, x1, tx = axis(new_w, w0)
    a = a.astype(np.float64)
    top = a[y0][:, x0] * (1 - tx)[None, :, None] + a[y0][:, x1] * tx[None, :, None]
    bot = a[y1][:, x0] * (1 - tx)[None, :, None] + a[y1][:, x1] * tx[None, :, None]
    out = top * (1 - ty)[:, None, None] + bot * ty[:, None, None]
    return np.floor(out + 0.5).clip(0, 255).astype(np.uint8)

In [ ]:
def letterbox(a, size=SIZE, pad_value=PAD_VALUE):
    """Return (640 x 640 image, scale ratio, (pad_x, pad_y))."""
    h0, w0 = a.shape[:2]
    r = min(size / w0, size / h0)
    new_w, new_h = round(w0 * r), round(h0 * r)
    if (new_w, new_h) != (w0, h0):
        a = resize_linear(a, new_w, new_h)
    pad_x = round((size - new_w) / 2 - 0.1)
    pad_y = round((size - new_h) / 2 - 0.1)
    canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = a
    return canvas, r, (pad_x, pad_y)

In [ ]:
lb, ratio, pad = letterbox(img)
print(f"ratio {ratio:.4f}, padding x {pad[0]}, y {pad[1]}")
display(Image.fromarray(lb).resize((320, 320)))

**Normalization and fixed point.** Pixel values 0–255 are normalized to 0–1 and converted to 16-bit fixed point
(Q6.10), the number format used by the accelerator. Channels are reordered from H×W×C to C×H×W.

In [ ]:
def to_q610(a):
    """Return a 3 x 640 x 640 int16 tensor in Q6.10."""
    return np.floor(a / 255.0 * 1024.0 + 0.5).astype(np.int16).transpose(2, 0, 1)

In [ ]:
q = to_q610(lb)
err = np.abs(q / 1024.0 - lb.transpose(2, 0, 1) / 255.0).max()
print(f"tensor {q.shape}, values {q.min()}..{q.max()}, max rounding error {err:.6f}")

## IMAGE INTO .TXT

The tensor is converted to the FPGA input format and saved as `frame_load.txt`, which `sending_image.tcl` sends
to the board. The scale ratio and padding are saved in `frame_meta.json` for `post_processing.ipynb`.

In [ ]:
def write_fpga_frame(q, path):
    rows = np.zeros((3 * SIZE, 1024), dtype="<i2")
    rows[:, :SIZE] = q.reshape(3 * SIZE, SIZE)
    buf = rows.tobytes()
    with open(path, "w") as f:
        for o in range(0, len(buf), 1024):
            words = np.frombuffer(buf[o:o + 1024], dtype="<u4")
            f.write("%08X %s\n" % (0x00100000 + o, " ".join("%08X" % w for w in words)))

In [ ]:
def save_meta(path, source, orig_w, orig_h, ratio, pad):
    """Save what post_processing.ipynb needs to map boxes back to the original image."""
    meta = dict(source=source, orig_w=int(orig_w), orig_h=int(orig_h), ratio=ratio, pad=[int(pad[0]), int(pad[1])])
    with open(path, "w") as f:
        json.dump(meta, f, indent=2)
    return meta

In [ ]:
write_fpga_frame(q, FRAME_FILE)
meta = save_meta(META_FILE, IMAGE_PATH.replace(os.sep, "/"), orig_w, orig_h, ratio, pad)
print(f"{FRAME_FILE} written")
print(json.dumps(meta, indent=2))

## CHECK

Confirms the frame file is complete before sending it, since sending takes a while.

In [ ]:
def check_frame(path, expected_lines=3 * SIZE * 2):
    """Raise an error if the frame file is incomplete or damaged."""
    with open(path) as f:
        lines = f.read().splitlines()
    if len(lines) != expected_lines or any(len(line.split()) != 257 for line in lines):
        raise ValueError(f"{path} is incomplete or damaged. Run this notebook again from the top.")

In [ ]:
check_frame(FRAME_FILE)
print(f"{IMAGE_NAME} is ready. Next: run sending_image.tcl in Vivado.")